# NeRF Reconstruction with Custom Adam

**Project title:** Optimization for 3D Scene Reconstruction: A Comparative Study of Optimizers, Losses, and Representations for NeRF and 3D Gaussian Splatting

**Group:** António Cruz, Duarte Cabrita

**Submission date:** May 18, 2026

---

# 1. Environment Setup

In [ ]:
# Libraries import

import math
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

In [ ]:
# Global variables

DEVICE = "cuda"
DATASET_PATH = "../data/tiny_nerf_data.npz"
RES = 100             # native resolution of tiny_nerf_data
N_SAMPLES = 64        # samples per ray
BATCH_RAYS = 1024
N_ITERATIONS = 5000
LR = 5e-4

In [ ]:
d = np.load(DATASET_PATH)

print({k: d[k].shape for k in d.files})

# 2. Data Loading

In [ ]:
data = np.load(DATASET_PATH)
images = torch.tensor(data["images"], dtype=torch.float32, device=DEVICE)   # [N, 100, 100, 3]
poses  = torch.tensor(data["poses"],  dtype=torch.float32, device=DEVICE)   # [N, 4, 4]
focal  = float(data["focal"])

# Hold out the last image as a test view; train on the rest.
train_images, train_poses = images[:-1], poses[:-1]
test_image,   test_pose   = images[-1],  poses[-1]

H, W = train_images.shape[1], train_images.shape[2]
print(f"Loaded {len(train_images)} training images at {H}x{W}, focal={focal:.2f}")

# 3. Ray Generation

In [ ]:
def get_rays(H, W, focal, pose):
    """Return ray origins and directions for every pixel in an HxW image."""
    i, j = torch.meshgrid(
        torch.arange(W, device=DEVICE).float(),
        torch.arange(H, device=DEVICE).float(),
        indexing="xy",
    )
    dirs = torch.stack([(i - W * 0.5) / focal, -(j - H * 0.5) / focal, -torch.ones_like(i)], dim=-1)
    rays_d = (dirs @ pose[:3, :3].T)
    rays_o = pose[:3, 3].expand(rays_d.shape)
    return rays_o, rays_d

# 4. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, num_freqs=10):
        super().__init__()
        self.freqs = 2.0 ** torch.arange(num_freqs).float().to(DEVICE)
        self.out_dim = 3 + 3 * 2 * num_freqs

    def forward(self, x):
        encs = [x]
        for f in self.freqs:
            encs += [torch.sin(f * x), torch.cos(f * x)]
        return torch.cat(encs, dim=-1)

# 5. NeRF MLP

In [ ]:
class TinyNeRF(nn.Module):
    def __init__(self, enc_dim, width=128, depth=4):
        super().__init__()
        layers = [nn.Linear(enc_dim, width), nn.ReLU()]
        for _ in range(depth - 1):
            layers += [nn.Linear(width, width), nn.ReLU()]
        self.trunk = nn.Sequential(*layers)
        self.density = nn.Linear(width, 1)
        self.rgb = nn.Linear(width, 3)

    def forward(self, x):
        h = self.trunk(x)
        sigma = torch.relu(self.density(h))[..., 0]
        c = torch.sigmoid(self.rgb(h))
        return sigma, c

# 6. Volume Rendering

In [ ]:
def render_rays(rays_o, rays_d, model, encoding, near=2.0, far=6.0, N=N_SAMPLES):
    t = torch.linspace(near, far, N, device=DEVICE)
    delta = (far - near) / N
    t = t + (torch.rand(rays_o.shape[0], N, device=DEVICE) - 0.5) * delta
    pts = rays_o[:, None] + rays_d[:, None] * t[:, :, None]
    sigma, c = model(encoding(pts.reshape(-1, 3)))
    sigma = sigma.reshape(rays_o.shape[0], N)
    c = c.reshape(rays_o.shape[0], N, 3)
    deltas = torch.cat([t[:, 1:] - t[:, :-1], torch.full_like(t[:, :1], 1e10)], dim=-1)
    alpha = 1.0 - torch.exp(-sigma * deltas)
    T = torch.cumprod(torch.cat([torch.ones_like(alpha[:, :1]), 1 - alpha + 1e-10], dim=-1), dim=-1)[:, :-1]
    w = T * alpha
    rgb = (w[..., None] * c).sum(dim=1)
    return rgb

# 7. Custom Adam Optimizer

In [ ]:
class MyAdam:
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = [p for p in params if p.requires_grad]
        self.lr, self.b1, self.b2, self.eps = lr, beta1, beta2, eps
        self.m = [torch.zeros_like(p) for p in self.params]
        self.v = [torch.zeros_like(p) for p in self.params]
        self.t = 0

    @torch.no_grad()
    def step(self):
        self.t += 1
        bc1 = 1 - self.b1 ** self.t
        bc2 = 1 - self.b2 ** self.t
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            g = p.grad
            m.mul_(self.b1).add_(g, alpha=1 - self.b1)
            v.mul_(self.b2).addcmul_(g, g, value=1 - self.b2)
            p.addcdiv_(m / bc1, v.div(bc2).sqrt().add_(self.eps), value=-self.lr)

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.detach_()
                p.grad.zero_()

# 8. Training Loop

In [ ]:
encoding = PositionalEncoding(num_freqs=10).to(DEVICE)
model = TinyNeRF(enc_dim=encoding.out_dim).to(DEVICE)
opt = MyAdam(list(model.parameters()), lr=LR)

losses = []
for it in tqdm(range(N_ITERATIONS)):
    img_idx = np.random.randint(len(train_images))
    target = train_images[img_idx]
    pose   = train_poses[img_idx]

    rays_o, rays_d = get_rays(H, W, focal, pose)
    rays_o = rays_o.reshape(-1, 3)
    rays_d = rays_d.reshape(-1, 3)
    target_flat = target.reshape(-1, 3)

    pix  = torch.randint(0, H * W, (BATCH_RAYS,), device=DEVICE)
    pred = render_rays(rays_o[pix], rays_d[pix], model, encoding)
    loss = ((pred - target_flat[pix]) ** 2).mean()

    opt.zero_grad()
    loss.backward()
    opt.step()
    losses.append(loss.item())

In [ ]:
# Compile the trained model for faster inference
# Pays one-time cost of ~15-30s, then every subsequent forward pass is 1.5-2x faster
model_c = torch.compile(model, mode="reduce-overhead", dynamic=False)

# Warmup: trigger compilation on a representative input shape so the first render isn't slow
with torch.inference_mode():
    dummy_enc = encoding(torch.zeros(H * W * N_SAMPLES, 3, device=DEVICE))
    _ = model_c(dummy_enc)
print("compile warmup done")

# 9. Plot the Loss Curve

In [ ]:
plt.figure(figsize=(8, 4))
plt.semilogy(losses)
plt.xlabel("Iteration"); plt.ylabel("L2 loss (log)")
plt.title("Training loss with custom Adam on Tiny NeRF")
plt.show()

# 10. Render a Novel View

In [ ]:
# Use the held-out test pose loaded in Cell 2
rays_o, rays_d = get_rays(H, W, focal, test_pose)
rays_o = rays_o.reshape(-1, 3)
rays_d = rays_d.reshape(-1, 3)

# Render in batches to avoid GPU OOM at full image resolution
with torch.no_grad():
    chunks = []
    for i in range(0, rays_o.shape[0], 4096):
        chunks.append(render_rays(rays_o[i:i+4096], rays_d[i:i+4096], model, encoding))
    pred = torch.cat(chunks, dim=0)

img = pred.reshape(H, W, 3).cpu().numpy()

plt.figure(figsize=(6, 6))
plt.imshow(np.clip(img, 0, 1))
plt.title("Novel view rendered by Tiny NeRF (custom Adam)")
plt.axis("off")
plt.show()

# 11. Side-By-Side Check

In [ ]:
gt = test_image.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(gt);                  axes[0].set_title("Ground truth (held-out view)")
axes[1].imshow(np.clip(img, 0, 1));  axes[1].set_title("Rendered by Tiny NeRF")
for a in axes: a.axis("off")
plt.show()

# 12. Compute PSNR

In [ ]:
mse = ((img - gt) ** 2).mean()
psnr = -10 * np.log10(mse)
print(f"PSNR: {psnr:.2f} dB")

# 13. Render View By Index

In [ ]:
def render_view(pose):
    rays_o, rays_d = get_rays(H, W, focal, pose)
    rays_o = rays_o.reshape(-1, 3)
    rays_d = rays_d.reshape(-1, 3)
    with torch.no_grad():
        chunks = []
        for i in range(0, rays_o.shape[0], 4096):
            chunks.append(render_rays(rays_o[i:i+4096], rays_d[i:i+4096], model, encoding))
        pred = torch.cat(chunks, dim=0)
    return pred.reshape(H, W, 3).cpu().numpy()

idx = 50   # Any index from 0 to 105
gt = images[idx].cpu().numpy()
rendered = render_view(poses[idx])

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(gt);                       axes[0].set_title(f"Ground truth (index {idx})")
axes[1].imshow(np.clip(rendered, 0, 1));  axes[1].set_title("Rendered")
for a in axes: a.axis("off")
plt.show()

mse = ((rendered - gt) ** 2).mean()
print(f"PSNR @ index {idx}: {-10 * np.log10(mse):.2f} dB")

# 14. Orbit Animation

In [ ]:
def render_view(pose):
    rays_o, rays_d = get_rays(H, W, focal, pose)
    rays_o = rays_o.reshape(-1, 3)
    rays_d = rays_d.reshape(-1, 3)
    with torch.no_grad():
        chunks = []
        for i in range(0, rays_o.shape[0], 4096):
            chunks.append(render_rays(rays_o[i:i+4096], rays_d[i:i+4096], model, encoding))
        pred = torch.cat(chunks, dim=0)
    return pred.reshape(H, W, 3).cpu().numpy()


def pose_spherical(theta_deg, phi_deg, radius):
    """Camera-to-world matrix at (radius, theta, phi), looking at the origin."""
    theta = np.deg2rad(theta_deg)
    phi   = np.deg2rad(phi_deg)

    c2w = torch.eye(4, device=DEVICE)
    c2w[2, 3] = radius

    R_phi = torch.eye(4, device=DEVICE)
    R_phi[1, 1] =  np.cos(phi);  R_phi[1, 2] = -np.sin(phi)
    R_phi[2, 1] =  np.sin(phi);  R_phi[2, 2] =  np.cos(phi)

    R_theta = torch.eye(4, device=DEVICE)
    R_theta[0, 0] =  np.cos(theta);  R_theta[0, 2] = -np.sin(theta)
    R_theta[2, 0] =  np.sin(theta);  R_theta[2, 2] =  np.cos(theta)

    c2w = R_theta @ R_phi @ c2w

    flip = torch.tensor([
        [-1, 0, 0, 0],
        [ 0, 0, 1, 0],
        [ 0, 1, 0, 0],
        [ 0, 0, 0, 1],
    ], dtype=torch.float32, device=DEVICE)
    return flip @ c2w

In [ ]:
import os, time, imageio
from IPython.display import Video

N_FRAMES, RADIUS, ELEVATION = 60, 4.0, -30.0

orbit_poses = [pose_spherical(t, ELEVATION, RADIUS)
               for t in np.linspace(0, 360, N_FRAMES + 1)[:-1]]

@torch.inference_mode()
def render_frame(pose):
    rays_o, rays_d = get_rays(H, W, focal, pose)
    rays_o = rays_o.reshape(-1, 3)
    rays_d = rays_d.reshape(-1, 3)
    pred = render_rays(rays_o, rays_d, model_c, encoding)   # model_c, not model
    return pred.reshape(H, W, 3)

t0 = time.time()
frames_gpu = torch.stack([render_frame(p) for p in tqdm(orbit_poses)])
frames = (frames_gpu.clamp(0, 1) * 255).to(torch.uint8).cpu().numpy()
print(f"Render took {time.time() - t0:.1f}s")

OUT_PATH = "../outputs/orbit.mp4"
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
imageio.mimsave(OUT_PATH, list(frames), fps=24, quality=8)
print(f"Saved to {OUT_PATH}")

Video(OUT_PATH, embed=True, width=400)